# 02 — Aprendizaje no supervisado con UNSW-NB15
**Equipo 1 — perfiles de tráfico, agrupamiento y anomalías**

**Integrantes:** Integrante 1 · Integrante 2 · Integrante 3 · Integrante 4  
**Asignatura:** Análisis de Datos · **Periodo:** 2026-2

Notebook independiente y reproducible. La etiqueta `attack` no entra al escalamiento, PCA, K-Means,
GMM, cálculo de perfiles ni puntuación de anomalías. Se conserva aparte solo para muestreo
estratificado representativo y para interpretación post-hoc, tal como permite el enunciado.

In [ ]:
from pathlib import Path
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.special import softmax
from IPython.display import display
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, average_precision_score,
    balanced_accuracy_score, calinski_harabasz_score, classification_report,
    completeness_score, confusion_matrix, davies_bouldin_score, f1_score,
    homogeneity_score, normalized_mutual_info_score, precision_recall_curve,
    precision_score, recall_score, roc_auc_score, roc_curve, silhouette_score,
    v_measure_score
)
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, RobustScaler, StandardScaler, label_binarize
from sklearn.svm import OneClassSVM, SVC

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 110, "savefig.dpi": 180})

MODE = "no_supervisado"
ROOT = Path.cwd()
VIS_DIR = ROOT / "visualizaciones" / MODE
RES_DIR = ROOT / "resultados" / MODE
CACHE_DIR = ROOT / ".cache_ml"
for folder in (VIS_DIR, RES_DIR, CACHE_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def locate_dataset(filename="UNSW-NB15_Dataset2.csv"):
    candidates = [ROOT / filename, ROOT / "data" / filename, ROOT.parent / filename]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"No se encontró {filename}. Ubíquelo en la carpeta del notebook, en data/ o en la carpeta padre."
    )


def savefig(name):
    path = VIS_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    plt.close()
    return path


def stratified_sample_indices(y, n, random_state=RANDOM_STATE):
    """Devuelve índices posicionales de una muestra estratificada reproducible."""
    y = np.asarray(y)
    if n >= len(y):
        return np.arange(len(y))
    idx, _ = train_test_split(
        np.arange(len(y)), train_size=n, stratify=y, random_state=random_state
    )
    return np.sort(idx)


def tune_f1_threshold(y_true, scores):
    """Ajusta el umbral solo con validación; scores altos significan 'resto de ataques'."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    best = int(np.nanargmax(f1))
    return float(thresholds[best]), float(f1[best])


def multiclass_scores(model, X, class_order):
    """Obtiene una matriz comparable de puntuaciones/probabilidades por clase."""
    if hasattr(model, "predict_proba"):
        raw = model.predict_proba(X)
    else:
        raw = softmax(model.decision_function(X), axis=1)
    model_classes = model.classes_
    order = [int(np.where(model_classes == c)[0][0]) for c in class_order]
    return raw[:, order]


def purity_score(y_true, y_cluster):
    tab = pd.crosstab(pd.Series(y_cluster, name="cluster"), pd.Series(y_true, name="clase"))
    return float(tab.max(axis=1).sum() / tab.to_numpy().sum())

print("Entorno preparado.")

## 1. Descripción del problema y preguntas

Se buscan perfiles matemáticos y comportamientos atípicos en 39 variables numéricas de conexión y
comportamiento. El CSV no contiene `proto`, `service`, `state` ni tráfico Normal; por ello los clusters
no se denominan "normal" o "ataque". Las etiquetas conocidas se consultan únicamente después del ajuste.

Preguntas: (1) ¿cuántos grupos justifican las métricas intrínsecas? (2) ¿K-Means o GMM produce mejor
compactación? (3) ¿qué variables caracterizan cada cluster? (4) ¿los grupos coinciden con los ataques?
(5) ¿qué registros aparecen como anómalos? (6) ¿qué preservan y qué pierden PCA y t-SNE?

In [ ]:
t0 = time.time()
DATA_PATH = locate_dataset()
df_raw = pd.read_csv(DATA_PATH)
TARGET = "attack"
FEATURES = [c for c in df_raw.columns if c != TARGET]
EXPECTED_CLASSES = ["Backdoor", "Probe", "Worm", "Generic", "DoS"]

assert TARGET in df_raw.columns
assert len(FEATURES) == 39
assert set(df_raw[TARGET].unique()) == set(EXPECTED_CLASSES)
assert all(pd.api.types.is_numeric_dtype(df_raw[c]) for c in FEATURES)

class_counts_raw = df_raw[TARGET].value_counts()
missing_total = int(df_raw.isna().sum().sum())
inf_total = int(np.isinf(df_raw[FEATURES].to_numpy()).sum())

print(f"Archivo: {DATA_PATH}")
print(f"Dimensión original: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Variables predictoras numéricas: {len(FEATURES)}")
print(f"Faltantes: {missing_total:,} | Infinitos: {inf_total:,}")
display(pd.DataFrame({"n": class_counts_raw, "%": (100 * class_counts_raw / len(df_raw)).round(2)}))

df = df_raw.copy()

## 2. EDA y análisis de variables disponibles

El archivo recibido **no contiene** `proto`, `service` ni `state`; tampoco contiene clase `Normal`.
No se inventan ni se codifican esas columnas. En su lugar se analizan las variables disponibles:

- **Volumen/tasa:** paquetes, bytes, cargas, pérdidas y medias.
- **Temporización:** duración, intervalos, jitter y tiempos TCP.
- **Estado/conexión:** TTL, ventanas TCP y `ct_state_ttl`.
- **Contexto histórico:** contadores `ct_*` de origen, destino, servicio y puertos.
- **Indicadores de aplicación:** FTP, HTTP, profundidad y longitud de respuesta.

In [ ]:
order = class_counts_raw.index.tolist()
fig, ax = plt.subplots(figsize=(8, 5))
colors = sns.color_palette("viridis", len(order))
sns.barplot(x=class_counts_raw.index, y=class_counts_raw.values, hue=class_counts_raw.index, ax=ax, palette=colors, legend=False)
ax.set(title="Distribución de ataques", xlabel="attack", ylabel="Registros")
ax.tick_params(axis="x", rotation=25)
for i, value in enumerate(class_counts_raw.values):
    ax.text(i, value, f"{value:,}\n{value/len(df_raw):.1%}", ha="center", va="bottom", fontsize=9)
savefig("01_eda_distribucion_clases.png")

In [ ]:
quality = pd.DataFrame({
    "indicador": ["Filas", "Faltantes", "Infinitos"],
    "valor": [len(df_raw), missing_total, inf_total]
})
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
sns.barplot(data=quality, x="indicador", y="valor", ax=axes[0], color="#0F766E")
axes[0].tick_params(axis="x", rotation=30)
axes[0].set(title="Calidad del archivo recibido", xlabel="", ylabel="Conteo")
axes[1].axis("off")
quality_text = (
    f"Filas: {len(df_raw):,}\n"
    f"Faltantes: {missing_total:,}\n"
    f"Infinitos: {inf_total:,}"
)
axes[1].text(0.05, 0.8, quality_text, fontsize=15, va="top", linespacing=1.6,
             bbox=dict(boxstyle="round,pad=0.8", facecolor="#ECFDF5", edgecolor="#0F766E"))
savefig("02_eda_calidad_datos.png")

In [ ]:
feature_groups = {
    "volumen_tasa": ["spkts", "dpkts", "sbytes", "dbytes", "rate", "sload", "dload", "sloss", "dloss", "smean", "dmean"],
    "tiempo": ["dur", "sinpkt", "dinpkt", "sjit", "djit", "tcprtt", "synack", "ackdat"],
    "estado_conexion": ["sttl", "dttl", "swin", "dwin", "ct_state_ttl"],
    "contexto": [c for c in FEATURES if c.startswith("ct_") and c != "ct_state_ttl"],
    "aplicacion": ["trans_depth", "response_body_len", "is_ftp_login", "ct_ftp_cmd", "ct_flw_http_mthd", "is_sm_ips_ports"]
}
# Perfil robusto: medianas por clase, estandarizadas por mediana/IQR global.
medians = df.groupby(TARGET)[FEATURES].median()
global_med = df[FEATURES].median()
iqr = (df[FEATURES].quantile(.75) - df[FEATURES].quantile(.25)).replace(0, 1)
robust_profile = ((medians - global_med) / iqr).clip(-5, 5)
profile_features = []
for group, cols in feature_groups.items():
    # dos variables con mayor variación entre clases por grupo
    profile_features += robust_profile[cols].var().nlargest(min(2, len(cols))).index.tolist()
profile_features = list(dict.fromkeys(profile_features))

plt.figure(figsize=(13, 5.5))
sns.heatmap(robust_profile[profile_features], cmap="vlag", center=0, annot=True, fmt=".1f", cbar_kws={"label": "Mediana robusta (IQR)"})
plt.title("Perfiles de conexión y comportamiento por ataque")
plt.xlabel("Variable disponible")
plt.ylabel("Clase")
savefig("03_eda_perfiles_comportamiento.png")
print("Variables destacadas por grupo:", profile_features)

In [ ]:
plot_features = ["dur", "sbytes", "dbytes", "rate", "sttl", "ct_state_ttl"]
sample_idx = stratified_sample_indices(df[TARGET], min(30000, len(df)))
eda_sample = df.iloc[sample_idx].copy()
long = eda_sample.melt(id_vars=TARGET, value_vars=plot_features, var_name="variable", value_name="valor")
long["log1p_valor"] = np.log1p(long["valor"].clip(lower=0))
g = sns.catplot(
    data=long, x=TARGET, y="log1p_valor", col="variable", col_wrap=3,
    kind="box", showfliers=False, sharey=False, height=3.4, aspect=1.2,
    hue=TARGET, palette="viridis", legend=False
)
g.set_axis_labels("Ataque", "log(1 + valor)")
g.set_titles("{col_name}")
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=30)
g.fig.suptitle("Distribuciones robustas de variables clave", y=1.02, fontsize=15)
plt.savefig(VIS_DIR / "04_eda_boxplots_log.png", bbox_inches="tight", dpi=180)
plt.show(); plt.close()

In [ ]:
corr = df[FEATURES].corr(method="spearman")
# Mostrar las 20 variables con mayor correlación media absoluta para conservar legibilidad.
selected_corr = corr.abs().mean().nlargest(20).index
plt.figure(figsize=(12, 10))
sns.heatmap(corr.loc[selected_corr, selected_corr], cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Correlación de Spearman — 20 variables más conectadas")
savefig("05_eda_correlacion_spearman.png")

# Porcentaje de outliers por regla IQR en todos los registros.
q1, q3 = df[FEATURES].quantile(.25), df[FEATURES].quantile(.75)
iqr_raw = q3 - q1
outlier_rate = (((df[FEATURES] < (q1 - 1.5 * iqr_raw)) | (df[FEATURES] > (q3 + 1.5 * iqr_raw))).mean() * 100).sort_values(ascending=False)
display(outlier_rate.head(10).rename("% outliers IQR").to_frame().round(2))

## 3. Preprocesamiento y muestreo sin usar la etiqueta como variable de entrada

Por costo computacional se toma una muestra estratificada reproducible de 25.000 registros;
`attack` solo determina las proporciones de muestreo y se separa inmediatamente. El pipeline recibe
exclusivamente las 39 columnas numéricas. Se aplica RobustScaler, que centra con la mediana y escala con el IQR para reducir el efecto de
los valores extremos. Finalmente, PCA conserva al menos 90 % de la varianza antes de K-Means y GMM.

**Algoritmos.** K-Means es rápido e interpretable, pero supone grupos aproximadamente esféricos y usa
distancias euclidianas. GMM permite pertenencia probabilística y componentes con distinta dispersión,
aunque es sensible a inicialización y a la especificación de covarianza.

In [ ]:
X_all = df[FEATURES].copy()
y_text_all = df[TARGET].copy()  # solo muestreo e interpretación post-hoc
label_encoder = LabelEncoder().fit(y_text_all)
y_posthoc_all = label_encoder.transform(y_text_all)
class_names = label_encoder.classes_.tolist()

cluster_idx = stratified_sample_indices(y_text_all, min(25000, len(y_text_all)), RANDOM_STATE + 11)
X_cluster_raw = X_all.iloc[cluster_idx].reset_index(drop=True)
attack_cluster = y_text_all.iloc[cluster_idx].reset_index(drop=True)
y_cluster_true = y_posthoc_all[cluster_idx]

# A partir de aquí, estas transformaciones y modelos solo reciben X_cluster_raw.
CLUSTER_FEATURES = list(FEATURES)
# RobustScaler centra con la mediana y escala con el IQR para reducir el efecto de los extremos.
cluster_scaler = RobustScaler().fit(X_cluster_raw[CLUSTER_FEATURES])
X_cluster_scaled = cluster_scaler.transform(X_cluster_raw[CLUSTER_FEATURES])
pca_probe = PCA(random_state=RANDOM_STATE).fit(X_cluster_scaled)
n_pca = max(2, int(np.searchsorted(np.cumsum(pca_probe.explained_variance_ratio_), 0.90) + 1))
pca_cluster = PCA(n_components=n_pca, random_state=RANDOM_STATE).fit(X_cluster_scaled)
X_cluster = pca_cluster.transform(X_cluster_scaled)
print(f"Muestra: {len(X_cluster):,} | variables originales: {len(FEATURES)} | PCA 90%: {X_cluster.shape[1]} componentes")
print("attack no está en X_cluster_raw:", TARGET not in X_cluster_raw.columns)